# Калибровка отказа и отдельная оценка holdout

Этот ноутбук показывает состояние исследовательского эксперимента и сохранённые метрики. Экспертной разметки пока нет. Он не создаёт экспертные метки и не запускает holdout при Run All.

Сначала завершите ручную работу из ноутбука 09 и экспортируйте проверенную версию в новый каталог `work/banking_review_v1`. По смысловым группам calibration и holdout должны быть разделены. Источники остаются research_only для США и английского языка.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.rag.banking_experiment import calibrate, load_review_bundle

BUNDLE = ROOT / 'work/banking_review_v1'
RUN_DIR = ROOT / 'work/banking_runs/bm25_v1'


## Готовность данных

Проверка ниже читает метаданные и метки обоих splits для проверки целостности и утечек, но не запускает поиск или подбор на holdout. Файловые проверки не подтверждают квалификацию или независимость человека. Эксперт должен действительно выполнить разметку; менять флаги у AI черновиков для прохождения проверки нельзя.

In [ ]:
if (BUNDLE / 'manifest.json').exists():
    try:
        manifest, splits = load_review_bundle(BUNDLE)
        display(pd.DataFrame([dict(split=name, queries=len(rows),
                                   answerable=sum(bool(row['relevant_document_ids']) for row in rows))
                              for name, rows in splits.items()]))
        print('Проверка формата пройдена. Источники остаются research_only.')
    except ValueError as error:
        print('Набор не готов:', error)
else:
    print('Ожидается реальная экспертная разметка и экспорт. Качество не оценивалось.')


## Подбор порога только на calibration

По умолчанию вычисление выключено. Для собственного запуска задайте новый RUN_DIR и включите RUN_CALIBRATION. Методы: bm25, dense, hybrid_rrf. Для dense и RRF используйте окружение requirements-hybrid с кешем закреплённой модели E5.

Инженерные ограничения по умолчанию: precision релевантной top-1 статьи не ниже 0.90 и доля принятых unanswerable не выше 0.05 на calibration. Это не подтверждённый банковский SLA и не точность сгенерированного ответа. Среди допустимых порогов выбирается максимальный supported recall, затем precision и более высокий порог. При отсутствии допустимого непустого решения сохраняется no_feasible_threshold. Ноль принятых запросов не считается качественным решением.

In [ ]:
RUN_CALIBRATION = False
METHOD = 'bm25'
MIN_PRECISION = 0.90
MAX_NEGATIVE_FAR = 0.05

if RUN_CALIBRATION:
    candidate = calibrate(BUNDLE, RUN_DIR, method=METHOD,
                          min_precision=MIN_PRECISION, max_negative_far=MAX_NEGATIVE_FAR)
    display(pd.Series(candidate))
else:
    print('Запуск калибровки выключен. Существующие результаты можно посмотреть ниже.')


In [ ]:
calibration_path = RUN_DIR / 'calibration.json'
if calibration_path.exists():
    calibration = json.loads(calibration_path.read_text(encoding='utf-8'))
    curve = pd.DataFrame(calibration['curve'])
    display(curve)
    display(pd.Series(calibration['selected'], dtype='object'))
    curve.plot(x='threshold', y=['coverage', 'supported_answer_precision',
                                'supported_answer_recall', 'unanswerable_false_accept_rate'],
               marker='o', figsize=(10, 5), title='Calibration only')
else:
    print('Калибровочные метрики ещё не рассчитаны.')


## Зафиксировать выбор и оценить holdout отдельно

Сравните варианты только по calibration и выберите один. candidate.json хранит метод, порог, ограничения, параметры, хеши кода и данных, версии библиотек. frozen.json связывает кандидата с сохранённой калибровкой. Изменение этих файлов блокирует оценку.

Отдельная команда, которую нужно запустить после фиксации решения:

```powershell
python scripts/run_banking_experiment.py holdout --run-dir work/banking_runs/bm25_v1
```

Она использует один зафиксированный порог. Перед поисковыми вызовами на holdout в каталоге BUNDLE создаётся holdout_started.json через exclusive create. Повторная попытка на этом bundle запрещена, включая другой RUN_DIR. Ошибка после начала оценки также расходует попытку. Не удаляйте marker для подбора по результату: для нового цикла нужны новые независимые вопросы. Это локальная защита от случайного повтора, не защита от человека с правом менять файлы. Сравнение нескольких методов на одном holdout потребует заранее определённого отдельного протокола.

Загрузка модели и проверка целостности происходят до marker. Поэтому отсутствие пакетов или повреждённая версия набора не расходуют попытку. Код проверки метаданных видит метки holdout, но не использует их для выбора порога.

In [ ]:
holdout_path = RUN_DIR / 'holdout.json'
if holdout_path.exists():
    holdout = json.loads(holdout_path.read_text(encoding='utf-8'))
    display(pd.DataFrame(holdout['abstention']))
    display(pd.Series({key: value for key, value in holdout['ranking'].items()
                       if key not in ['answerable_query_results', 'language_metrics']}))
else:
    print('Holdout не оценён. Run All не запускает оценку.')


## Как читать результаты

Recall@5 усредняется только по вопросам с релевантными статьями, после объединения фрагментов одной статьи. MRR здесь учитывает полный список найденных кандидатов и не называется MRR@5. Precision при отказе означает долю принятых запросов, где top-1 статья релевантна. Он не доказывает, что статья содержит полный и актуальный ответ.

Unanswerable false accept rate — доля вопросов без ответа, которые были приняты. Unsafe accept rate в отчёте дополнительно учитывает неправильную top-1 статью у answerable, но это название технического показателя, а не полноценная оценка безопасности. На небольшом наборе оценки нестабильны; выводы о реальных клиентах требуют репрезентативной выборки. Новый результат не переключает API и не разрешает источники для обслуживания.

После реальной retrieval оценки отдельно проверяются выбор действий агентом, цитаты, корректность уточнений и передачи оператору. Метрики агента из демонстрационных сценариев не заменяют эту проверку.